In [1]:
# ==============================================================================
# ĐỒ ÁN MÔN PHÂN TÍCH DỮ LIỆU MẠNG XÃ HỘI
# Sinh viên thực hiện: LÊ XUÂN THÀNH
# Nhiệm vụ: Tải dữ liệu, tiền xử lý, trích xuất Sub-graph và thống kê Topology
# ==============================================================================

import networkx as nx
import pandas as pd
import urllib.request
import gzip
import shutil
import os
import pickle
import random
from pathlib import Path

# ============================================================
# PATH CONFIGURATION
# ============================================================
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"
DATA_DIR.mkdir(parents=True, exist_ok=True)

GRAPH_PATH = DATA_DIR / "amazon_graph.gpickle"

# Dữ liệu gốc tải từ SNAP sẽ được lưu tạm trong data/processed.
# Khi push GitHub, có thể bỏ qua 2 file raw này bằng .gitignore.
URL = "https://snap.stanford.edu/data/bigdata/communities/com-amazon.ungraph.txt.gz"
GZ_PATH = DATA_DIR / "com-amazon.ungraph.txt.gz"
TXT_PATH = DATA_DIR / "com-amazon.ungraph.txt"

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"DATA_DIR     = {DATA_DIR}")
print("Đã import thành công các thư viện!")

# ============================================================
# 1. TẢI VÀ GIẢI NÉN DỮ LIỆU SNAP
# ============================================================
print("Đang tải dữ liệu từ Stanford SNAP...")

try:
    if not GZ_PATH.exists() and not TXT_PATH.exists():
        urllib.request.urlretrieve(URL, GZ_PATH)
        print("Tải xong! Đang giải nén dữ liệu...")

        with gzip.open(GZ_PATH, "rb") as f_in:
            with open(TXT_PATH, "wb") as f_out:
                shutil.copyfileobj(f_in, f_out)

        print("Giải nén hoàn tất!")
    else:
        print("File dữ liệu đã tồn tại, bỏ qua bước tải.")
except urllib.error.HTTPError as e:
    print(f"[LỖI TẢI DỮ LIỆU] Link Stanford SNAP có thể đang bảo trì. Lỗi: {e}")
    print("Hãy thử lại sau hoặc tải file thủ công từ SNAP.")

# ============================================================
# 2. ĐỌC DỮ LIỆU VÀO NETWORKX
# ============================================================
print("Đang xây dựng đồ thị từ tập dữ liệu gốc...")

G_full = nx.read_edgelist(
    TXT_PATH,
    comments="#",
    create_using=nx.Graph(),
    nodetype=int
)

print("--- THỐNG KÊ ĐỒ THỊ GỐC ---")
print(f"Số lượng Node (Sản phẩm): {G_full.number_of_nodes():,}")
print(f"Số lượng Edge (Lượt mua chung): {G_full.number_of_edges():,}")

# ============================================================
# 3. TRÍCH XUẤT SUB-GRAPH 8,000 NODES BẰNG BFS
# ============================================================
print("\nĐang tìm sản phẩm trung tâm để làm điểm bắt đầu...")

degrees = dict(G_full.degree())
start_node = max(degrees, key=degrees.get)
print(f"Sản phẩm trung tâm (Node ID: {start_node}) có {degrees[start_node]} lượt mua kèm.")

print("Đang trích xuất đồ thị con gồm 8,000 nodes bằng BFS...")
TARGET_NODES = 8000

bfs_edges = list(nx.bfs_edges(G_full, source=start_node, depth_limit=4))
sub_nodes = {start_node}

for u, v in bfs_edges:
    if len(sub_nodes) >= TARGET_NODES:
        break
    sub_nodes.add(v)

G_sub = G_full.subgraph(sub_nodes).copy()
print("Trích xuất sub-graph thành công!")

# ============================================================
# 4. THỐNG KÊ NETWORK TOPOLOGY
# ============================================================
print("\n" + "=" * 50)
print("THỐNG KÊ NETWORK TOPOLOGY")
print("=" * 50)

num_nodes = G_sub.number_of_nodes()
num_edges = G_sub.number_of_edges()
density = nx.density(G_sub)

print(f"1. Tổng số Nodes: {num_nodes:,}")
print(f"2. Tổng số Edges: {num_edges:,}")
print(f"3. Density: {density:.5f}")

if nx.is_connected(G_sub):
    print("4. Đồ thị liên thông hoàn toàn.")
    print("Đang tính Average Shortest Path Length...")
    avg_shortest_path = nx.average_shortest_path_length(G_sub)
else:
    print("4. Đồ thị KHÔNG liên thông hoàn toàn.")
    largest_cc = max(nx.connected_components(G_sub), key=len)
    G_largest_cc = G_sub.subgraph(largest_cc)
    print(f"Đang tính Average Shortest Path trên cụm lớn nhất ({len(largest_cc)} nodes)...")
    avg_shortest_path = nx.average_shortest_path_length(G_largest_cc)

print(f"5. Average Shortest Path: {avg_shortest_path:.4f}")
print("=" * 50)

# ============================================================
# 5. LƯU FILE GRAPH CHO CÁC PHẦN SAU
# ============================================================
print(f"\nĐang lưu đồ thị vào file: {GRAPH_PATH}")

with open(GRAPH_PATH, "wb") as f:
    pickle.dump(G_sub, f, pickle.HIGHEST_PROTOCOL)

print("LƯU THÀNH CÔNG!")

PROJECT_ROOT = /content
DATA_DIR     = /content/data/processed
Đã import thành công các thư viện!
Đang tải dữ liệu từ Stanford SNAP...
Tải xong! Đang giải nén dữ liệu...
Giải nén hoàn tất!
Đang xây dựng đồ thị từ tập dữ liệu gốc...
--- THỐNG KÊ ĐỒ THỊ GỐC ---
Số lượng Node (Sản phẩm): 334,863
Số lượng Edge (Lượt mua chung): 925,872

Đang tìm sản phẩm trung tâm để làm điểm bắt đầu...
Sản phẩm trung tâm (Node ID: 548091) có 549 lượt mua kèm.
Đang trích xuất đồ thị con gồm 8,000 nodes bằng BFS...
Trích xuất sub-graph thành công!

THỐNG KÊ NETWORK TOPOLOGY
1. Tổng số Nodes: 8,000
2. Tổng số Edges: 19,903
3. Density: 0.00062
4. Đồ thị liên thông hoàn toàn.
Đang tính Average Shortest Path Length...
5. Average Shortest Path: 6.2674

Đang lưu đồ thị vào file: /content/data/processed/amazon_graph.gpickle
LƯU THÀNH CÔNG!
